In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
import time

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
from keras.losses import CategoricalCrossentropy
import csv

from loss import *
from SoftQuantizeLayer import SoftQuantizeLayer
from AnnealingScheduler import AnnealingScheduler
from DAE_DataGenerator import OptimizedDataGenerator as DAE_OptDG
from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from dataset_utils import add_noise, quantize_manual
from plotting import *


2026-08-30 18:15:33.087643: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [19]:
import models
import importlib
importlib.reload(models)
from models import *

thresholds = [1,240,800]
model = create_SQDAE_transformer_slim_model(input_shape=(16,16,2),   
                             patch_size=(3,4),        
                             embed_dim=64,           
                             num_heads=4,            
                             ff_dim=128,              
                             num_layers=4,            
                             dropout=0.1,        
                             threshold_offset=0,
                             n_bits=2,
                             initial_thresholds=thresholds,
                             final_outputs=3         
                            )
model.compile(
    # optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    optimizer=tf.keras.optimizers.Nadam(learning_rate=2e-3, clipnorm=1.0),
    # loss=MeanSquaredError()
    # loss=CustomSSIMClass(xrange = xrange, log_standardize = False),
    # loss=CustomSSIM(logxrange = xrange, 
    #                 log_standardize = True)
    loss = CategoricalCrossentropy(reduction = 'sum', from_logits=True
                                  )
    # run_eagerly=True
)

model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 raw_input (InputLayer)         [(None, 16, 16, 2)]  0           []                               
                                                                                                  
 soft_quantizer_output (SoftQua  (None, 16, 16, 2)   4           ['raw_input[0][0]']              
 ntizeLayer)                                                                                      
                                                                                                  
 patch_extractor_10 (PatchExtra  (None, None, 24)    0           ['soft_quantizer_output[0][0]']  
 ctor)                                                                                            
                                                                                            

In [17]:
import importlib
import DAE_DataGenerator
import dataset_utils
import SoftQuantizeLayer
import AnnealingScheduler

importlib.reload(DAE_DataGenerator)
importlib.reload(dataset_utils)
importlib.reload(SoftQuantizeLayer)
importlib.reload(AnnealingScheduler)

from AnnealingScheduler import AnnealingScheduler
from DAE_DataGenerator import OptimizedDataGenerator
from dataset_utils import add_noise, quantize_manual
#from dataprep import *
# from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from loss import *

In [1]:
dataset_base_dir = "/uscms/home/bweiss/nobackup/smart-pixels/"
tfrecords_base_dir = os.path.join(dataset_base_dir, "tfrecords")

# dataset_dir_train = os.path.join(dataset_base_dir, "/dataset_3sr_16x16_50x12P5_parquets", 'train/')
# dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets", 'test/')

dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'train_contained/')
dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'test_contained/')

# tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'SQDAE_80eN_catgoricalOut_Target_0_7_23_134_404_867_1748')
# tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'SQDAE_80eN_catgoricalOut_Target_0_7_23_134_404_867_1748')

tfrecords_dir_train   = os.path.join(tfrecords_base_dir, "TFR_train",'SQDAE_3srCI_80eN_catgoricalOut_Target_0_30_845')
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'SQDAE_3srCI_80eN_catgoricalOut_Target_0_30_845')



# tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'3sr_16x16_80eN_raw')
# tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'3sr_16x16_80eN_raw')


print(dataset_dir_val)
batch_size = 5000
val_batch_size = 5000
train_file_size = 80
val_file_size = 20

#### Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=False,
    # seed=42,
    quantize=False,
    bitwise_categorical = True,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=False,
    # seed=42,
    quantize=False,
    bitwise_categorical = True,
)

/uscms/home/bweiss/nobackup/smart-pixels/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/test_contained/


NameError: name 'OptimizedDataGenerator' is not defined

In [20]:
from datetime import datetime
tname = 'SQDAE_SlimTransformer_Categorical_2bit_80eN_Target_0_30_845_TEST'#0_7_23_134_404_867_1748'
fingerprint = '%08x' % random.randrange(16**8)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./SQ_DAE/model-{fingerprint}-{tname}-checkpoints'
os.makedirs(base_dir, exist_ok=True)  
checkpoint_filepath = base_dir + '/weights.{epoch:02d}-t{loss:.4f}-v{val_loss:.4f}.hdf5'
print("fingerprint: ", fingerprint)

fingerprint:  bb57b09d


In [23]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
import csv

early_stopping_patience = 1000

class CustomModelCheckpoint(ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
        if len(checkpoints) > 1:
            checkpoints.sort(key=lambda x: os.path.getmtime(os.path.join(base_dir,x)))
            for checkpoint in checkpoints[:-1]:
                os.remove(os.path.join(base_dir, checkpoint))

class SoftQuantizeLoggerCallback(Callback):
    def __init__(self, log_filepath, layer_name="soft_quantizer_output"):
        super().__init__()
        self.log_filepath = log_filepath
        self.layer_name = layer_name
        self.header_written = False

    def on_train_begin(self, logs=None):
        os.makedirs(os.path.dirname(self.log_filepath), exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        try:
            layer = self.model.get_layer(self.layer_name)
            if not hasattr(layer, 'n_bits'):
                 print(f"\nWarning: Layer '{self.layer_name}' is not a SoftQuantizeLayer. Skipping logging.")
                 return
        except ValueError:
            print(f"\nWarning: Layer '{self.layer_name}' not found in the model. Skipping logging.")
            return

        if not self.header_written:
            num_levels = layer.num_levels
            num_thresholds = num_levels - 1
            
            header = ['epoch', 'k']
            # header.extend([f'level_{i}' for i in range(num_levels)])
            header.extend([f'threshold_{i}' for i in range(num_thresholds)])

            # header.append('raw_first_level')
            # header.extend([f'raw_log_level_delta_{i}' for i in range(num_levels - 1)])
            # header.append('raw_first_threshold')
            # if hasattr(layer, 'log_threshold_deltas'):
            #     header.extend([f'raw_log_threshold_delta_{i}' for i in range(num_thresholds - 1)])

            with open(self.log_filepath, mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(header)
            self.header_written = True

        k_val = layer.k.numpy().item()
        # levels = layer.levels.numpy().tolist()
        thresholds = layer.thresholds.numpy().tolist()
        
        # first_level = layer.first_level.numpy().item()
        # log_level_deltas = layer.log_level_deltas.numpy().tolist()
        # first_threshold = layer.first_threshold.numpy().item()
        
        row_data = [epoch, k_val]
        # row_data.extend(levels)
        row_data.extend(thresholds)
        # row_data.append(first_level)
        # row_data.extend(log_level_deltas)
        # row_data.append(first_threshold)
        
        # if hasattr(layer, 'log_threshold_deltas'):
        #     log_threshold_deltas = layer.log_threshold_deltas.numpy().tolist()
        #     row_data.extend(log_threshold_deltas)
        
        with open(self.log_filepath, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(row_data)


csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)
scheduler_callback = AnnealingScheduler(
    schedule='cosine',  
    target_layer_name='soft_quantizer_output', 
    initial_k=1.0,
    final_k=400.0, 
    verbose=1      
)

quantizer_logger = SoftQuantizeLoggerCallback(
    log_filepath=f"{base_dir}/soft_quantizer_state_log.csv", # New, more descriptive filename
    layer_name="soft_quantizer_output"
)

es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = CustomModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True,
    save_freq='epoch',
    verbose=1
)


In [24]:
history = model.fit(
        x=training_generator,
        validation_data=validation_generator,
        callbacks=[es, mcp, csv_logger,scheduler_callback, quantizer_logger],
        epochs=5000,
        shuffle=False,
        verbose=1
    )


Epoch 1: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.3133, 240.3133, 800.3134
	Tau: [120.15664 399.50003 560.00006]
Epoch 1/5000


2026-08-24 18:40:29.112949: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-08-24 18:40:29.493892: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8800


84/84 [==============================] - ETA: 0s - loss: 711383.6250
Epoch 1: val_loss improved from inf to 413161.75000, saving model to ./SQ_DAE/model-bb57b09d-SQDAE_SlimTransformer_Categorical_2bit_80eN_Target_0_30_845_TEST-checkpoints/weights.01-t711383.6250-v413161.7500.hdf5
84/84 [==============================] - 94s 904ms/step - loss: 711383.6250 - val_loss: 413161.7500

Epoch 2: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.5119, 268.6056, 866.7585
	Tau: [134.30281 432.62335 598.15295]
Epoch 2/5000
84/84 [==============================] - ETA: 0s - loss: 367284.7188
Epoch 2: val_loss improved from 413161.75000 to 302372.53125, saving model to ./SQ_DAE/model-bb57b09d-SQDAE_SlimTransformer_Categorical_2bit_80eN_Target_0_30_845_TEST-checkpoints/weights.02-t367284.7188-v302372.5312.hdf5
84/84 [==============================] - 23s 274ms/step - loss: 367284.7188 - val_loss: 302372.5312

Epoch 3: Annealing 'k' set to 1.0001
	Levels: 0.0000, 1.000

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



84/84 [==============================] - ETA: 0s - loss: 93640.7109
Epoch 1190: val_loss did not improve from 101612.58594
84/84 [==============================] - 24s 287ms/step - loss: 93640.7109 - val_loss: 102104.6094

Epoch 1191: Annealing 'k' set to 27.5411
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 145.7094, 313.7318, 884.6427
	Tau: [156.86589 369.46667 570.91095]
Epoch 1191/5000
84/84 [==============================] - ETA: 0s - loss: 93608.8828
Epoch 1191: val_loss did not improve from 101612.58594
84/84 [==============================] - 25s 292ms/step - loss: 93608.8828 - val_loss: 102119.0781

Epoch 1192: Annealing 'k' set to 27.5836
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 145.4654, 313.0924, 884.6790
	Tau: [156.5462  369.60678 571.58655]
Epoch 1192/5000
84/84 [==============================] - ETA: 0s - loss: 93641.9531
Epoch 1192: val_loss did not improve from 101612.58594
84/84 [==============================] - 22s 259ms/step - loss: 93641.9531 - 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

